### Import libraries

In [2]:
pip install pandas

  Using cached pandas-3.0.5-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.5.3-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.5-cp312-cp312-win_amd64.whl (9.8 MB)
Using cached numpy-2.5.3-cp312-cp312-win_amd64.whl (12.6 MB)
Using cached tzdata-2026.3-py2.py3-none-any.whl (348 kB)

   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Pandas version:", pd.__version__)
print("Notebook environment ready.")

Pandas version: 3.0.5
Notebook environment ready.


### Locate the raw dataset

In [3]:
current_dir = Path.cwd()

possible_paths = [
    current_dir / "data" / "raw" / "online_retail_II.xlsx",
    current_dir.parent / "data" / "raw" / "online_retail_II.xlsx",
]

DATA_PATH = None

for path in possible_paths:
    if path.exists():
        DATA_PATH = path.resolve()
        break

print("Current working directory:")
print(current_dir)

print("\nDataset path:")
print(DATA_PATH)

print("\nFile exists:")
print(DATA_PATH is not None)

Current working directory:
c:\New folder\Hands on Projects\Data-Analysis\customer-sales-inventory-operations-optimization\python

Dataset path:
C:\New folder\Hands on Projects\Data-Analysis\customer-sales-inventory-operations-optimization\data\raw\online_retail_II.xlsx

File exists:
True


### Check file size

In [4]:
file_size_mb = DATA_PATH.stat().st_size / (1024 ** 2)

print(f"Raw dataset size: {file_size_mb:,.2f} MB")

Raw dataset size: 43.51 MB


### Inspect Excel sheet names

In [5]:
excel_file = pd.ExcelFile(DATA_PATH)

print("Number of worksheets:", len(excel_file.sheet_names))
print("Worksheet names:")

for sheet in excel_file.sheet_names:
    print("-", sheet)

Number of worksheets: 2
Worksheet names:
- Year 2009-2010
- Year 2010-2011


### Read only the first few rows of each sheet

In [6]:
for sheet in excel_file.sheet_names:
    
    sample = pd.read_excel(
        DATA_PATH,
        sheet_name=sheet,
        nrows=5
    )
    
    print("=" * 80)
    print("SHEET:", sheet)
    print("=" * 80)
    
    display(sample)

SHEET: Year 2009-2010


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


SHEET: Year 2010-2011


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


### Inspect column names

In [7]:
for sheet in excel_file.sheet_names:
    
    sample = pd.read_excel(
        DATA_PATH,
        sheet_name=sheet,
        nrows=5
    )
    
    print(f"\nSheet: {sheet}")
    
    for i, column in enumerate(sample.columns, start=1):
        print(f"{i}. {column}")


Sheet: Year 2009-2010
1. Invoice
2. StockCode
3. Description
4. Quantity
5. InvoiceDate
6. Price
7. Customer ID
8. Country

Sheet: Year 2010-2011
1. Invoice
2. StockCode
3. Description
4. Quantity
5. InvoiceDate
6. Price
7. Customer ID
8. Country


### Load both years

In [8]:
raw_sheets = {}

for sheet in excel_file.sheet_names:
    
    print(f"Loading: {sheet} ...")
    
    df = pd.read_excel(
        DATA_PATH,
        sheet_name=sheet
    )
    
    raw_sheets[sheet] = df
    
    print(f"Loaded {len(df):,} rows")

Loading: Year 2009-2010 ...
Loaded 525,461 rows
Loading: Year 2010-2011 ...
Loaded 541,910 rows


### Source reconciliation

In [9]:
total_rows = 0

for sheet, df in raw_sheets.items():
    
    rows, columns = df.shape
    
    print(
        f"{sheet}: "
        f"{rows:,} rows × {columns} columns"
    )
    
    total_rows += rows

print("-" * 50)
print(f"TOTAL SOURCE ROWS: {total_rows:,}")

Year 2009-2010: 525,461 rows × 8 columns
Year 2010-2011: 541,910 rows × 8 columns
--------------------------------------------------
TOTAL SOURCE ROWS: 1,067,371


### Check schemas

In [10]:
for sheet, df in raw_sheets.items():
    
    print("=" * 80)
    print("SHEET:", sheet)
    print("=" * 80)
    
    print(df.dtypes)
    print()

SHEET: Year 2009-2010
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

SHEET: Year 2010-2011
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object



### Date boundaries

In [11]:
for sheet, df in raw_sheets.items():

    print("=" * 60)
    print(f"SHEET: {sheet}")
    print("=" * 60)

    print("Minimum Invoice Date:", df["InvoiceDate"].min())
    print("Maximum Invoice Date:", df["InvoiceDate"].max())
    print()

SHEET: Year 2009-2010
Minimum Invoice Date: 2009-12-01 07:45:00
Maximum Invoice Date: 2010-12-09 20:01:00

SHEET: Year 2010-2011
Minimum Invoice Date: 2010-12-01 08:26:00
Maximum Invoice Date: 2011-12-09 12:50:00



In [12]:
overall_min_date = min(
    df["InvoiceDate"].min()
    for df in raw_sheets.values()
)

overall_max_date = max(
    df["InvoiceDate"].max()
    for df in raw_sheets.values()
)

print("Overall transaction period:")
print("Start:", overall_min_date)
print("End:  ", overall_max_date)

Overall transaction period:
Start: 2009-12-01 07:45:00
End:   2011-12-09 12:50:00
